In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2
from pyproj import Transformer

In [ ]:
# ── 경로 설정 ─────────────────────────────────────────────────────────────
BESSEL_CSV  = Path('./dataset/processed/raw_parquet/rain/서울시 강우량계 위치정보(베셀(bessel) 좌표계).csv')
SEWER_NODE  = Path('./dataset/processed/cleaned/sewer_node.parquet')
SEWER_IDX   = Path('./dataset/features/sewer_node_index.parquet')
OUT_STATION = Path('./dataset/processed/rain_station_coords.parquet')
OUT_MAPPING = Path('./dataset/processed/sewer_rain_mapping.parquet')

## Step 1. 베셀 좌표 CSV → WGS84 (위경도) 변환

 좌표 체계:
   CSV X좌표 = northing(m) × 100  (cm 단위 northing)

   CSV Y좌표 = easting(m)  × 100  (cm 단위 easting)
   
   좌표 기준: EPSG:5181 (Korea 2000 / Central Belt, GRS80)

In [ ]:
def bessel_to_wgs84(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, encoding='cp949')

    transformer = Transformer.from_crs(5181, 4326, always_xy=True)

    lats, lons = [], []
    for _, row in df.iterrows():
        easting_m  = row['설치위치(Y좌표)'] / 100   # CSV Y = easting
        northing_m = row['설치위치(X좌표)'] / 100   # CSV X = northing
        lon, lat   = transformer.transform(easting_m, northing_m)
        lats.append(round(lat, 6))
        lons.append(round(lon, 6))

    return pd.DataFrame({
        'station_id'   : df['강우량계 코드'].astype(int),
        'station_name' : df['강우량계명'].astype(str),
        'district_code': df['구청 코드'].astype(int),
        'district_name': df['구청명'].astype(str),
        'address'      : df['주소'].astype(str),
        'lat'          : lats,
        'lon'          : lons,
    })


# Step 2. 하수관로 노드 → 최근접 강우 관측소 매핑

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    d = lambda x: radians(x)
    a = sin(d(lat2-lat1)/2)**2 + cos(d(lat1))*cos(d(lat2))*sin(d(lon2-lon1)/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))


def map_sewer_to_rain(sewer_node_path: Path,
                      sewer_idx_path: Path,
                      rain_stations: pd.DataFrame) -> pd.DataFrame:
    sewer_node = pd.read_parquet(sewer_node_path)
    sewer_idx  = pd.read_parquet(sewer_idx_path)

    # 학습에 사용되는 노드만 (위경도 있는 것)
    used = sewer_node[sewer_node['sensor_id'].isin(sewer_idx['sensor_id'])]
    used = used.dropna(subset=['lat', 'lon'])

    rows = []
    for _, sn in used.iterrows():
        dists = rain_stations.apply(
            lambda r: haversine(sn['lat'], sn['lon'], r['lat'], r['lon']), axis=1
        )
        best = rain_stations.loc[dists.idxmin()]
        rows.append({
            'sensor_id'        : sn['sensor_id'],
            'rain_station_id'  : int(best['station_id']),
            'rain_station_name': best['station_name'],
            'distance_m'       : round(dists.min(), 1),
        })

    return pd.DataFrame(rows)

## Step 3. 실행

In [ ]:
if __name__ == '__main__':
    print('Step 1: 베셀 좌표 변환...')
    rain_stations = bessel_to_wgs84(BESSEL_CSV)
    rain_stations.to_parquet(OUT_STATION, index=False)
    print(f'  → {OUT_STATION}  ({len(rain_stations)}개 관측소)')

    print('Step 2: 하수관로-강우관측소 매핑...')
    mapping = map_sewer_to_rain(SEWER_NODE, SEWER_IDX, rain_stations)
    mapping.to_parquet(OUT_MAPPING, index=False)
    print(f'  → {OUT_MAPPING}  ({len(mapping)}개 노드 매핑)')

    print('\n=== 거리 분포 요약 ===')
    print(mapping['distance_m'].describe().round(0))

    print('\n=== 관측소별 담당 노드 수 ===')
    print(mapping.groupby('rain_station_name')['sensor_id']
          .count().sort_values(ascending=False).to_string())

## Step 4. 강수량 데이터 전처리 설정

```
원시 데이터 문제:
  - 타임스탬프 불규칙: 관측소마다 :09, :19, :08, :18... 등 정시 기준 아님
  - 이상치: 453.5mm/10분 등 물리적 불가능한 값 존재
  - 관측소 수 변화: 43개(2022-01) → 48개(2022-05 이후)

전처리 순서:
  1. 이상치 제거  (RAIN_UPPER_MM 초과 → NaN)
  2. 관측소별 10분 리샘플  (:00, :10, :20... 정시 정렬, sum 집계)
  3. 정규화  (mm/10min → mm/hr, 100mm/hr 기준 min-max → [0, 1])
```

In [ ]:
import gc
import pyarrow as pa
import pyarrow.parquet as pq

RAW_RAIN_DIR   = Path('./dataset/processed/raw_parquet/rain')
CLEAN_RAIN_DIR = Path('./dataset/processed/cleaned/rain')
CLEAN_RAIN_DIR.mkdir(parents=True, exist_ok=True)

# ── 전처리 파라미터 ─────────────────────────────────────────────
RAIN_UPPER_MM   = 50.0   # 10분 강우 물리적 상한 (≈ 300mm/hr, 이 초과는 센서 오류)
RAIN_MMHR_NORM  = 100.0  # 정규화 기준: 100mm/hr (gnn_config 기준)
# rainfall_norm = clip(rainfall_mm × 6 / 100, 0, 1)
#   ↑ 10분 → 시간당 환산(×6), 100mm/hr 기준 min-max

CLEAN_SCHEMA = pa.schema([
    ('station_id',    pa.int32()),
    ('timestamp',     pa.timestamp('ns')),
    ('rainfall_mm',   pa.float32()),   # 10분 강우량 (이상치 제거 후)
    ('rainfall_norm', pa.float32()),   # 정규화값 [0, 1]
])

## Step 5. 전처리 함수 정의

### 처리 흐름
```
raw parquet (월별)
  └─ 이상치 제거: rainfall_mm > 50 → NaN, < 0 → NaN
  └─ 10분 리샘플 (관측소별):
       - freq='10min', label='left', closed='left'
       - agg: sum  → 같은 10분 버킷 내 값 합산
       - NaN → 0  (결측 = 무강우로 처리)
  └─ 정규화: rainfall_norm = clip(rainfall_mm × 6 / 100, 0, 1)
  └─ 저장: cleaned/rain/{filename}.parquet
```

In [ ]:
def process_rain_file(src_path: Path, out_path: Path,
                      upper_mm: float, mmhr_norm: float) -> dict:
    """
    월별 raw 강수량 parquet → 이상치 처리 + 10분 리샘플 + 정규화 → 저장

    Returns: 처리 결과 요약 dict
    """
    fname = src_path.stem

    # ── 읽기 ──────────────────────────────────────────
    df = pd.read_parquet(src_path, columns=['station_id', 'timestamp', 'rainfall_mm'])
    n_raw = len(df)

    # ── 이상치 제거 ───────────────────────────────────
    n_over  = int((df['rainfall_mm'] > upper_mm).sum())
    n_under = int((df['rainfall_mm'] < 0).sum())
    df.loc[df['rainfall_mm'] > upper_mm, 'rainfall_mm'] = np.nan
    df.loc[df['rainfall_mm'] < 0,        'rainfall_mm'] = np.nan

    # ── 관측소별 10분 리샘플 ──────────────────────────
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')

    parts = []
    for sid, grp in df.groupby('station_id'):
        resampled = (grp['rainfall_mm']
                     .resample('10min', label='left', closed='left')
                     .sum(min_count=1))  # min_count=1: 전부 NaN이면 NaN 유지
        resampled = resampled.fillna(0.0)  # 결측 버킷 = 무강우(0)

        sub = resampled.reset_index()
        sub.columns = ['timestamp', 'rainfall_mm']
        sub['station_id'] = sid
        parts.append(sub)

    df_clean = pd.concat(parts, ignore_index=True)

    # ── 정규화 ────────────────────────────────────────
    # mm/10min → mm/hr (×6), 100mm/hr 기준 min-max → [0, 1]
    df_clean['rainfall_norm'] = (
        (df_clean['rainfall_mm'] * 6.0 / mmhr_norm)
        .clip(0.0, 1.0)
        .astype('float32')
    )
    df_clean['rainfall_mm']  = df_clean['rainfall_mm'].astype('float32')
    df_clean['station_id']   = df_clean['station_id'].astype('int32')
    df_clean['timestamp']    = df_clean['timestamp'].astype('datetime64[ns]')

    # ── 저장 ─────────────────────────────────────────
    table = pa.Table.from_pandas(
        df_clean[['station_id', 'timestamp', 'rainfall_mm', 'rainfall_norm']],
        schema=CLEAN_SCHEMA,
        preserve_index=False,
    )
    pq.write_table(table, out_path, compression='snappy')

    return {
        'file'    : fname,
        'n_raw'   : n_raw,
        'n_clean' : len(df_clean),
        'n_over'  : n_over,
        'n_under' : n_under,
        'max_mm'  : float(df_clean['rainfall_mm'].max()),
        'size_kb' : out_path.stat().st_size // 1024,
    }

## Step 6. 전체 44개 파일 일괄 처리 및 검증

In [ ]:
import time

raw_files = sorted(RAW_RAIN_DIR.glob('*.parquet'))
print(f'처리 대상: {len(raw_files)}개 파일')
print()

records = []
t_start = time.time()

for src in raw_files:
    out = CLEAN_RAIN_DIR / src.name
    if out.exists():
        print(f'  SKIP: {src.name}')
        continue

    t0  = time.time()
    res = process_rain_file(src, out, RAIN_UPPER_MM, RAIN_MMHR_NORM)
    res['sec'] = round(time.time() - t0, 1)
    records.append(res)

    flag = f'  [이상치 {res["n_over"]}건]' if res['n_over'] else ''
    print(f'  OK: {src.name}  '
          f'raw={res["n_raw"]:,} → clean={res["n_clean"]:,}행  '
          f'{res["size_kb"]}KB  {res["sec"]}s{flag}')

    del res
    gc.collect()

print()
print(f'총 소요: {time.time()-t_start:.1f}초')

# ── 결과 요약 ─────────────────────────────────────
if records:
    df_log = pd.DataFrame(records)
    print()
    print('=== 이상치 제거 요약 ===')
    print(df_log[df_log['n_over'] > 0][['file','n_over','max_mm']].to_string(index=False))
    print()
    print(f'총 raw 행수   : {df_log["n_raw"].sum():,}')
    print(f'총 clean 행수 : {df_log["n_clean"].sum():,}')
    print(f'총 이상치 제거: {df_log["n_over"].sum():,}건')

In [ ]:
# ── 결과 검증: 파일 스키마 확인 ─────────────────────────────────
sample_path = CLEAN_RAIN_DIR / '서울시_강우량_정보_2024년07월.parquet'

print('=== 2024년07월 정제 파일 메타데이터 ===')
if sample_path.exists():
    schema = pq.read_schema(sample_path)
    print(f'path   : {sample_path}')
    print(f'size KB: {sample_path.stat().st_size // 1024:,}')
    print(f'columns: {schema.names}')
else:
    print(f'파일 없음: {sample_path}')


## Step 7. Train / Val / Test 분할

```
Train : 2024-01-01 ~ 2024-12-31  (하수관로·도로노면과 동일 기준)
Val   : 2025-01-01 ~ 2025-05-31
Test  : 2025-06-01 ~ 2025-08-31

출력 경로: dataset/features/rain/{split}/rain_{split}.parquet
컬럼: station_id (int32) | timestamp (datetime64[ns])
     | rainfall_mm (float32) | rainfall_norm (float32)

In [ ]:
SPLITS = {
    'train': ('2024-01-01', '2024-12-31 23:59:59'),
    'val'  : ('2025-01-01', '2025-05-31 23:59:59'),
    'test' : ('2025-06-01', '2025-08-31 23:59:59'),
}

FEAT_RAIN_BASE = Path('./dataset/features/rain')
for s in SPLITS:
    (FEAT_RAIN_BASE / s).mkdir(parents=True, exist_ok=True)

# ── 정제된 월별 파일 전체 스트리밍 병합 후 분할 저장 ─────────────
clean_files = sorted(CLEAN_RAIN_DIR.glob('*.parquet'))
print(f'정제 파일 수: {len(clean_files)}개')
print()

# split별 ParquetWriter 생성
writers = {
    s: pq.ParquetWriter(FEAT_RAIN_BASE / s / f'rain_{s}.parquet', CLEAN_SCHEMA)
    for s in SPLITS
}
counts = {s: 0 for s in SPLITS}

for f in clean_files:
    df = pd.read_parquet(f)
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    for split, (start, end) in SPLITS.items():
        sub = df[(df['timestamp'] >= start) & (df['timestamp'] <= end)]
        if sub.empty:
            continue

        table = pa.Table.from_pandas(
            sub[['station_id', 'timestamp', 'rainfall_mm', 'rainfall_norm']],
            schema=CLEAN_SCHEMA,
            preserve_index=False,
        )
        writers[split].write_table(table)
        counts[split] += len(sub)

    del df
    gc.collect()

for split, w in writers.items():
    w.close()

print('=== 분할 결과 ===')
for split, n in counts.items():
    path = FEAT_RAIN_BASE / split / f'rain_{split}.parquet'
    size_kb = path.stat().st_size // 1024
    print(f'  {split:5s}: {n:>10,}행  {size_kb:>6,} KB  → {path}')

## Step 8. 최종 검증

In [ ]:
print('=' * 60)
print('강수량 전처리 파이프라인 최종 검증')
print('=' * 60)

CHECKS = [
    (Path('./dataset/processed/rain_station_coords.parquet'),  '강우관측소 좌표 (WGS84)'),
    (Path('./dataset/processed/sewer_rain_mapping.parquet'),   '하수관로→강우관측소 매핑'),
    (FEAT_RAIN_BASE / 'train' / 'rain_train.parquet',          'Rain Train (2024)'),
    (FEAT_RAIN_BASE / 'val'   / 'rain_val.parquet',            'Rain Val   (2025-01~05)'),
    (FEAT_RAIN_BASE / 'test'  / 'rain_test.parquet',           'Rain Test  (2025-06~08)'),
]

all_ok = True
for path, desc in CHECKS:
    exists = path.exists()
    size   = path.stat().st_size // 1024 if exists else 0
    mark   = 'OK' if exists else 'MISSING'
    print(f'  {mark:<8} {desc:<35}  {size:>6,} KB')
    if not exists:
        all_ok = False

print()
print('=== Split별 스키마 확인 ===')
for split in ('train', 'val', 'test'):
    path = FEAT_RAIN_BASE / split / f'rain_{split}.parquet'
    if not path.exists():
        print(f'  [{split}] 파일 없음')
        all_ok = False
        continue

    schema = pq.read_schema(path)
    has_required = {'station_id', 'timestamp', 'rainfall_mm', 'rainfall_norm'}.issubset(schema.names)
    print(f'  [{split}]  {"OK" if has_required else "MISSING"}  columns={schema.names}')
    if not has_required:
        all_ok = False

print()
print('완료 — 강수량 데이터 전처리 준비 완료' if all_ok else '미완료 항목 확인 필요')
print()
print('다음 단계:')
print('  rain_train/val/test.parquet + sewer_rain_mapping.parquet')
print('  → preprocessing_rain_join.ipynb에서 하수관로 피처에 rainfall_norm 추가')
print('  → F_SEWER: 9 → 10')
